In [12]:
import json
import random
from pathlib import Path


def sample_file_groups_with_code(json_path, base_dir, n=10, seed=42):
    with open(json_path, "r", encoding="utf-8") as f:
        preds = json.load(f)

    base_dir = Path(base_dir)
    rng = random.Random(seed)

    def is_green(pred):
        labels = pred["predicted_labels"]
        return (
            labels["interaction"] == ["yes"]
            and labels["outcome"][-1] == "static"
        )

    def is_top2_orange(pred):
        labels = pred["predicted_labels"]
        return (
            labels["entities"] == ["synthesized_image"]
            and labels["interaction"] == ["no"]
            and labels["outcome"] == ["visual", "time_based"]
        )

    def is_top9(pred):
        labels = pred["predicted_labels"]
        return (
            labels["entities"] == ["processed_image", "processed_audio", "randomness"]
            and labels["interaction"] == ["yes"]
            and labels["outcome"] == ["visual", "auditory", "time_based"]
        )

    def sample_with_code(items):
        items = items.copy()
        rng.shuffle(items)

        selected = []

        for pred in items:
            file_path = pred["file_path"]
            full_path = base_dir / file_path

            if not full_path.exists():
                continue

            src_code = full_path.read_text(encoding="utf-8", errors="ignore").strip()

            if not src_code:
                continue

            selected.append({
                "file_path": file_path,
                "predicted_labels": pred["predicted_labels"],
                "label_combination": pred["label_combination"],
                "src_code": src_code
            })

            if len(selected) == n:
                break

        return selected

    green = [pred for pred in preds if is_green(pred)]
    top2_orange = [pred for pred in preds if is_top2_orange(pred)]
    top9 = [pred for pred in preds if is_top9(pred)]

    return {
        "green": sample_with_code(green),
        "top2_orange": sample_with_code(top2_orange),
        "top9": sample_with_code(top9)
    }

In [13]:
json_path = "/home/rpinter/art-in-swh/data/clean_artworks_label_prediction.json"
base_dir = "/home/rpinter/links/projects/def-baudry/shared/data/artworks/src/"

samples = sample_file_groups_with_code(
    json_path=json_path,
    base_dir=base_dir,
    n=10,
    seed=42
)

samples

{'green': [{'file_path': 'MadeCurler/website/CodingChallenges/CC_090_dithering/P5/sketch.js',
   'predicted_labels': {'entities': ['processed_image', 'synthesized_image'],
    'interaction': ['yes'],
    'outcome': ['visual', 'static']},
   'label_combination': "entities=['processed_image', 'synthesized_image'] | interaction=['yes'] | outcome=['visual', 'static']",
   'src_code': 'let kitten;\n\nfunction preload() {\n  kitten = loadImage("data/kitten.jpg");\n}\n\nfunction setup() {\n  createCanvas(1024, 512);\n\n  image(kitten, 0, 0);\n  makeDithered(kitten, 1);\n  image(kitten, 512, 0);\n  // Apply gray filter to the whole canvas\n  filter(GRAY);\n}\n\nfunction imageIndex(img, x, y) {\n  return 4 * (x + y * img.width);\n}\n\nfunction getColorAtindex(img, x, y) {\n  let idx = imageIndex(img, x, y);\n  let pix = img.pixels;\n  let red = pix[idx];\n  let green = pix[idx + 1];\n  let blue = pix[idx + 2];\n  let alpha = pix[idx + 3];\n  return color(red, green, blue, alpha);\n}\n\nfunction

In [14]:
with open("sampled_files_with_code.json", "w", encoding="utf-8") as f:
    json.dump(samples, f, indent=2, ensure_ascii=False)